# Resume Tailoring LLM — Fine-Tuning with LoRA

**Author:** Akshay Pillalamarri

**What this does:** Fine-tunes Microsoft Phi-2 using LoRA (Parameter-Efficient Fine-Tuning) to take a base resume + job description as input and generate a tailored resume as output.

**Why LoRA:** Instead of training all 2.7B parameters of Phi-2, we only train a small adapter (~1% of parameters). Runs on free T4 GPU in Colab.

**Estimated time:** 60-90 minutes total

## Before running:
1. Go to Runtime → Change runtime type → T4 GPU
2. Run cells in order

## Step 1: Install Dependencies

In [ ]:
!pip install -q transformers==4.38.0 peft==0.8.2 datasets==2.17.0 accelerate==0.27.2 bitsandbytes==0.42.0 trl==0.7.10
!pip install -q gradio huggingface_hub

## Step 2: Verify GPU is Available

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
print('GPU name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')
print('CUDA version:', torch.version.cuda)

## Step 3: Load Training Dataset

Upload `resume_dataset.json` to Colab files first (left sidebar → upload icon).

In [ ]:
import json
from datasets import Dataset

with open('resume_dataset.json', 'r') as f:
    raw_data = json.load(f)

print(f'Total training examples: {len(raw_data)}')
print('\nFirst example preview:')
print(json.dumps(raw_data[0], indent=2)[:500])

def format_example(ex):
    text = f"""### Instruction:
Tailor the following resume to match the job description provided. Optimize keywords, reorder skills by relevance, and emphasize matching experience.

### Job Description:
{ex['job_description']}

### Base Resume:
{ex['base_resume']}

### Tailored Resume:
{ex['tailored_resume']}"""
    return {'text': text}

dataset = Dataset.from_list(raw_data).map(format_example)
print(f'\nDataset prepared: {len(dataset)} examples')

## Step 4: Load Phi-2 Base Model with 4-bit Quantization

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = 'microsoft/phi-2'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)

print('Model loaded successfully')
print(f'Model device: {next(model.parameters()).device}')

## Step 5: Configure LoRA Adapter

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'dense'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Step 6: Tokenize Dataset

In [ ]:
def tokenize(example):
    result = tokenizer(
        example['text'],
        truncation=True,
        max_length=1024,
        padding='max_length',
    )
    result['labels'] = result['input_ids'].copy()
    return result

tokenized_dataset = dataset.map(tokenize, remove_columns=dataset.column_names)
print(f'Tokenized {len(tokenized_dataset)} examples')

## Step 7: Train the Model

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir='./phi2-resume-tailor',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=5,
    save_strategy='epoch',
    save_total_limit=1,
    report_to='none',
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

trainer.train()
print('\nTraining complete!')

## Step 8: Save the Fine-Tuned Adapter

In [ ]:
model.save_pretrained('./phi2-resume-tailor-final')
tokenizer.save_pretrained('./phi2-resume-tailor-final')
print('Model adapter saved')

## Step 9: Test Inference

In [ ]:
def tailor_resume(job_description, base_resume, max_new_tokens=400):
    prompt = f"""### Instruction:
Tailor the following resume to match the job description provided. Optimize keywords, reorder skills by relevance, and emphasize matching experience.

### Job Description:
{job_description}

### Base Resume:
{base_resume}

### Tailored Resume:
"""
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result.split('### Tailored Resume:')[-1].strip()

test_jd = 'Senior AI Engineer role focused on LangGraph, multi-agent systems, RAG pipelines, and Python.'
test_resume = 'Akshay Pillalamarri, Software Developer at Cloud Nexus. Skills: Python, Java, Selenium, Salesforce.'

result = tailor_resume(test_jd, test_resume)
print('TAILORED RESUME OUTPUT:\n')
print(result)

## Step 10: Push to Hugging Face Hub

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
model.push_to_hub('akshaypillalamarri/phi2-resume-tailor-lora')
tokenizer.push_to_hub('akshaypillalamarri/phi2-resume-tailor-lora')
print('Pushed to Hugging Face Hub')